In [2]:
!pip install jaxlib
!pip install jax
!pip install git+https://github.com/deepmind/dm-haiku
!pip install gymnasium
!pip install gymnasium[box2d]
!pip install optax
!pip install matplotlib
!pip install chex
!pip install gymnasium[classic_control]

  Cloning https://github.com/deepmind/dm-haiku to /tmp/pip-req-build-y4aw3amq
  Running command git clone --filter=blob:none --quiet https://github.com/deepmind/dm-haiku /tmp/pip-req-build-y4aw3amq
  Resolved https://github.com/deepmind/dm-haiku to commit 0d53d52d82a7a18c805a01596b5058d8a9ea2f93
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 70.2 MB/s eta 0:00:00


In [ ]:
import copy
from shutil import rmtree # deleting directories
import random
import collections # useful data structures
import numpy as np
import gym # reinforcement learning environments
from gym.wrappers import RecordVideo
import jax
import jax.numpy as jnp # jax numpy
import haiku as hk # jax neural network library
import optax # jax optimizer library
import matplotlib.pyplot as plt # graph plotting library
from IPython.display import HTML
from base64 import b64encode
import chex

# Hide warnings
import warnings
warnings.filterwarnings('ignore')

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


упражнение 1

In [ ]:
def linear_policy(params, obs):
  dot_product_result = jax.numpy.dot(params, obs)
  action = jax.lax.select(
      dot_product_result > 0,
      1,  #True
      0,  #False
  )
  return action

упражнение 2

In [ ]:
def run_episode(env):
  episode_return = 0  # счётчик для отслеживания наград
  done = False  # изначально эпизод не завершён
  params = jnp.array([1,-2,2,-1])  # фиксированные параметры политики

  obs, info = env.reset()  # получаем начальное наблюдение из среды

  while not done:  # цикл до завершения эпизода

    action = linear_policy(params, obs)  # вычисляем действие с помощью линейной политики
    action = np.array(action)  # преобразуем действие из политики в np.array

    obs, reward, terminated, truncated, info = env.step(action)  # делаем шаг в среде
    done = terminated or truncated  # эпизод завершён, если шест упал или достигнут лимит шагов

    episode_return += reward  # добавляем награду к возврату эпизода

  return episode_return

упражнение 3

In [ ]:
def random_policy_search_choose_action(
    key,
    params,
    actor_state,
    obs,
    evaluation=False
):

  best_action = linear_policy(params.best, obs)  # действие, используем лучшие параметры

  current_action = linear_policy(params.current, obs)  # действие, используем текущие параметры

  action = jax.lax.select(
      evaluation,  # если evaluation == True, используем лучшие параметры, иначе текущие
      best_action,  # результат, когда условие истинно (evaluation == True)
      current_action  # результат, когда условие ложно (evaluation == False)
  )

  return action, actor_state

упражнение 4

In [ ]:
def get_new_random_weights(random_key, old_weights, minval=-2.0, maxval=2.0):
    new_weights_shape = old_weights.shape  # форма новых весов
    new_weights_dtype = old_weights.dtype  # тип данных новых весов

    new_params = jax.random.uniform(
        key=random_key,           # ключ случайности
        shape=new_weights_shape,  # форма массива (4 числа)
        minval=minval,            # минимальное значение (-2.0)
        maxval=maxval,            # максимальное значение (2.0)
        dtype=new_weights_dtype   # тип данных
    )

    return new_params

упражнение 5

In [ ]:
def random_policy_search_learn(key, params, learn_state, memory):
    best_params = params.best
    current_params = params.current

    current_average_episode_return = memory  # memory содержит средний возврат за эпизод
    best_average_episode_return = learn_state.best_average_episode_return

    # Проверяем, лучше ли текущие параметры, чем лучшие
    best_params = jax.lax.select(
        current_average_episode_return > best_average_episode_return,  # условие: текущий возврат больше лучшего?
        current_params,  # если True: обновляем лучшие параметры текущими
        best_params      # если False: оставляем старые лучшие параметры
    )

    best_average_episode_return = jax.lax.select(
        current_average_episode_return > best_average_episode_return,  # условие: текущий возврат больше лучшего?
        current_average_episode_return,  # если True: обновляем лучший средний возврат
        best_average_episode_return       # если False: оставляем старый лучший возврат
    )

    # Генерируем новые случайные параметры
    new_params = get_new_random_weights(key, current_params)

    # Упаковываем веса в NamedTuple RandomPolicySearchParams
    params = RandomPolicySearchParams(current=new_params, best=best_params)

    return params, RandomPolicyLearnState(best_average_episode_return)

упражнение 6

In [ ]:
def compute_weighted_log_prob(action_prob, episode_return):

    log_prob = jax.numpy.log(action_prob)  # вычисляем логарифм вероятности действия

    weighted_log_prob = log_prob * episode_return  # умножаем логарифм вероятности на отдачу эпизода

    return weighted_log_prob

упражнение 7

In [ ]:
def compute_rewards_to_go(rewards):

    rewards_to_go = []

    n = len(rewards)

    # Способ с накоплением суммы с конца
    cumulative_sum = 0
    for t in range(n - 1, -1, -1):  # идём с конца к началу
        cumulative_sum += rewards[t]  # добавляем текущую награду
        rewards_to_go.append(cumulative_sum)  # сохраняем оставшуюся награду

    # Разворачиваем список, чтобы порядок был правильным
    rewards_to_go = rewards_to_go[::-1]

    return rewards_to_go

упражнение 8

In [ ]:
def sample_action(random_key, logits):

  # Преобразуем логиты в вероятности с помощью softmax
  probabilities = jax.nn.softmax(logits)

  # Случайно выбираем действие из категориального распределения
  action = jax.random.categorical(key=random_key, logits=logits)

  return action

упражнение 9

In [ ]:
def policy_gradient_loss(action, logits, reward_to_go):

  all_action_probs = jax.nn.softmax(logits)  # преобразуем логиты в вероятности

  action_prob = all_action_probs[action]  # извлекаем вероятность заданного действия

  weighted_log_prob = compute_weighted_log_prob(action_prob, reward_to_go)  # вычисляем взвешенный логарифм вероятности

  loss = - weighted_log_prob  # отрицательное значение, потому что нам нужен градиентный `подъём`

  return loss

упражнение 10

In [ ]:
def select_greedy_action(q_values):

  action = jax.numpy.argmax(q_values)  # находим индекс с максимальным Q-значением

  return action

упражнение 11

In [ ]:
def compute_squared_error(pred, target):

  squared_error = jax.numpy.square(pred - target)

  return squared_error

упражнение 12

In [ ]:
# Bellman target
def compute_bellman_target(reward, done, next_q_values, gamma=0.99):

  # Находим максимальное Q-значение для следующего состояния
  max_next_q_value = jax.numpy.max(next_q_values)

  # Используем jax.lax.select для условия: если done == 1.0, то не добавляем max_next_q_value
  bellman_target = jax.lax.select(
      done == 1.0,                       # условие: терминальное ли состояние?
      reward,                            # если True: цель = просто награда
      reward + gamma * max_next_q_value  # если False: цель = награда + дисконтированное след. Q
  )

  return bellman_target

упражнение 13

In [ ]:
def q_learning_loss(q_values, action, reward, done, next_q_values):

    chosen_action_q_value = q_values[action]  # Q-значение выбранного действия, используя индексацию массива
    bellman_target = compute_bellman_target(reward, done, next_q_values)  # вычисляем цель Беллмана
    squared_error = jax.numpy.square(chosen_action_q_value - bellman_target)  # квадратичная ошибка

    return squared_error

упражнение 14

In [ ]:
def select_random_action(key, num_actions):

    # Генерируем случайное целое число от 0 до num_actions-1 включительно
    action = jax.random.randint(key=key, shape=(), minval=0, maxval=num_actions)

    return action

упражнение 15

In [ ]:
EPSILON_DECAY_TIMESTEPS = 3000 # decay epsilon over 3000 timesteps
EPSILON_MIN = 0.1 # 10% exploration

def get_epsilon(num_timesteps):

  # Линейное затухание epsilon от 1.0 до EPSILON_MIN за EPSILON_DECAY_TIMESTEPS шагов
  epsilon = 1.0 - (num_timesteps / EPSILON_DECAY_TIMESTEPS) * (1.0 - EPSILON_MIN)

  # Ограничиваем epsilon снизу значением EPSILON_MIN
  epsilon = jax.lax.select(
      epsilon < EPSILON_MIN,
      EPSILON_MIN,  # если меньше минимума, то устанавливаем в минимум
      epsilon       # иначе оставляем epsilon без изменений
  )

  return epsilon

упражнение 16

In [ ]:
def select_epsilon_greedy_action(key, q_values, num_timesteps):
    num_actions = len(q_values)  # количество доступных действий

    epsilon = get_epsilon(num_timesteps)  # получаем значение эпсилон

    # Генерируем случайное число от 0 до 1 и проверяем, меньше ли оно эпсилон
    random_number = jax.random.uniform(key, shape=())
    should_explore = random_number < epsilon  # логическое выражение: исследовать, если случайное число меньше эпсилон

    action = jax.lax.select(
        should_explore,
        select_random_action(key, num_actions),  # если исследовать: выбираем случайное действие
        select_greedy_action(q_values)           # если жадное: выбираем действие с максимальным Q-значением
    )

    return action